# youtube_channels.csv 전처리를 위한 파일입니다.

## 컬럼 목록

| 컬럼명 | 타입 | 설명 |
|---|---|---|
| `channel_name` | string | 유튜브 채널 이름. `" - Topic"` 형태는 유튜브가 자동 생성한 아티스트 채널로 실제 운영 크리에이터가 아님 |
| `category` | string | 채널 카테고리 (게임, 음식/요리/레시피, 뉴스/정치/사회 등 22종). `미분류`는 카테고리 미지정 채널 |
| `subscriber_raw` | int | 구독자 수 원시값 (수집 당시 API 반환값 그대로) |
| `subscriber_count` | int | 구독자 수 정제값 (분석에 사용) |
| `total_views_raw` | int | 채널 누적 조회수 원시값 |
| `total_views` | int | 채널 누적 조회수 정제값 |
| `video_count_raw` | string | 업로드 영상 수 원시값. `"3개"` 처럼 단위 문자 포함된 문자열 |
| `video_count` | int | 업로드 영상 수 정제값 (숫자만 추출). 0이면 영상 없는 채널 |
| `created_date` | datetime | 채널 개설일시 (UTC 기준). null이면 API에서 수집 실패 |
| `latest_video_date` | datetime | 가장 최근 영상 업로드 일시. null이면 영상이 없거나 비공개 |
| `days_since_latest_video` | float | 마지막 영상 업로드 후 경과 일수. `latest_video_date`가 null이면 함께 null |
| `is_churned` | float | **이탈 라벨** (1.0 = 이탈, 0.0 = 활성). null이면 라벨 미지정 (모델 학습 시 제외 필요) |
| `youtube_channel_url` | string | 채널 URL (`https://www.youtube.com/channel/UC...`) |
| `youtube_channel_id` | string | 채널 고유 ID (`UC`로 시작하는 24자리 문자열). YouTube Data API v3 호출 키 |

---

## 전처리 시 제거 대상

| 제거 조건 | 해당 행 수 | 이유 |
|---|---|---|
| `channel_name`이 `- Topic` 또는 `Release - Topic`으로 끝남 | 894개 | 유튜브 자동 생성 아티스트 채널 — 실제 크리에이터 아님 |
| `video_count == 0` | 370개 | 영상이 없어 활동 지표 계산 불가 |
| 채널 생성일 기준 6개월 미만 or `created_date` null | 40개 | 이탈 여부 판단을 위한 최소 관찰 기간 미충족 |


# 1. 라이브러리 , 데이터 로드

In [5]:
import pandas as pd

df = pd.read_csv('data/raw/youtube_channels.csv')

df.head()

,channel_name,category,subscriber_raw,subscriber_count,total_views_raw,total_views,video_count_raw,video_count,created_date,latest_video_date,days_since_latest_video,is_churned,youtube_channel_url,youtube_channel_id
0,장정숙,뉴스/정치/사회,1,1,492,492,3개,3,2016-06-08 15:54:31,2019-12-11 14:58:53,2347.0,1.0,https://www.youtube.com/channel/UCo3Yj54VtkEvQ...,UCo3Yj54VtkEvQX9cLHKklzw
1,심해 생존일지스타트의,게임,1,1,940,940,3개,3,2017-08-07 02:58:34,2017-11-11 21:03:32,3107.0,1.0,https://www.youtube.com/channel/UCyABUa7lzjsV4...,UCyABUa7lzjsV4o5PfSFqymw
2,의원실김정우,뉴스/정치/사회,1,1,41,41,1개,1,2016-06-04 22:29:52,2017-03-08 15:56:28,3355.0,1.0,https://www.youtube.com/channel/UCGPHpVMxzO2rc...,UCGPHpVMxzO2rc5SqhGIc2tg
3,장석주,미분류,1,1,0,0,0개,0,2013-10-08 16:03:43,NaN,NaN,NaN,https://www.youtube.com/channel/UCDfRXnMKZmokf...,UCDfRXnMKZmokf5Jtl8K6nHQ
4,- Topic,미분류,1,1,0,0,0개,0,2018-05-15 22:25:55,NaN,NaN,NaN,https://www.youtube.com/channel/UCra9lyIlp4Q3e...,UCra9lyIlp4Q3eE-cDiEiJbw


# 2. 데이터 현황

In [6]:
# null 현황 알아보자
df.isnull().sum()

channel_name                 0
category                     0
subscriber_raw               0
subscriber_count             0
total_views_raw              0
total_views                  0
video_count_raw              0
video_count                  0
created_date                 2
latest_video_date          282
days_since_latest_video    282
is_churned                 282
youtube_channel_url          2
youtube_channel_id           2
dtype: int64

In [7]:
# 카테고리 분포
df["category"].value_counts().to_string()

'category\n미분류          2462\n게임            861\n키즈/어린이        797\n음악/댄스/가수      706\n취미/라이프        680\nTV/방송         621\nBJ/인물/연예인     564\n뉴스/정치/사회      443\n음식/요리/레시피     386\n패션/미용         349\n교육/강의         263\n스포츠/운동        212\n영화/만화/애니      210\n회사/오피셜        185\n국내/해외/여행      176\n애완/반려동물       166\n자동차           163\nIT/기술/컴퓨터      97\n주식/경제/부동산      66\n해외             17\nBJ인물연예인         1\n취미              1'

# - Topic/Release-Topic 채널 삭제

유튜브는 음원등록 아티스트는 자동으로 채널을 생성함

이들을 이탈 예측 대상에서 제외

In [ ]:
# 제거 대상 미리 확인
mask_topic = df["channel_name"].str.endswith("- Topic", na=False)

print(f"제거 대상: {mask_topic.sum():,}개")
print()
df[mask_topic][["channel_name", "category"]].head(10)

In [ ]:
# 실제 제거
df = df[~mask_topic].reset_index(drop=True)

print(f"제거 후 행 수: {len(df):,}")

# video_count == 0 제거

In [ ]:
# 제거 대상 확인
mask_no_video = df["video_count"] == 0

print(f"제거 대상: {mask_no_video.sum():,}개")
df[mask_no_video][["channel_name", "video_count_raw", "video_count"]].head(5)


In [ ]:
# 실제 제거
df = df[~mask_no_video].reset_index(drop=True)

print(f"제거 후 행 수: {len(df):,}")


# 채널 생성 6개월 이하 제거

In [ ]:
# 날짜 파싱 (파싱 실패 → NaT)
df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce", utc=True)

# null 확인
print(f"created_date null: {df['created_date'].isnull().sum()}개")


In [ ]:
# 기준일: 오늘로부터 6개월 전
import pandas as pd

reference_date = pd.Timestamp.now(tz="UTC")
cutoff = reference_date - pd.DateOffset(months=6)
print(f"기준일: {reference_date.date()}")
print(f"컷오프: {cutoff.date()} (이 날짜 이후 개설된 채널 제거)")


In [ ]:
# 제거 대상 확인 (null 포함)
mask_new_channel = df["created_date"].isnull() | (df["created_date"] > cutoff)

print(f"제거 대상: {mask_new_channel.sum():,}개")
df[mask_new_channel][["channel_name", "created_date"]].head(5)


# 결과 확인

In [ ]:
# 실제 제거
df = df[~mask_new_channel].reset_index(drop=True)

print(f"제거 후 행 수: {len(df):,}")

In [ ]:
print("=== 전처리 결과 요약 ===")
print(f"  원본:  9,426행")
print(f"  최종:  {len(df):,}행")
print(f"  제거:  {9426 - len(df):,}행")


In [ ]:
# 잔여 null 확인
print("=== 잔여 null 현황 ===")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
# is_churned 라벨 분포
print("=== is_churned 분포 ===")
print(df["is_churned"].value_counts(dropna=False))
print()
print("※ NaN은 라벨 미지정 — 모델 학습 시 제외 필요")


In [ ]:
df.to_csv("youtube_channels_cleaned.csv", index=False, encoding="utf-8-sig")
print("저장 완료: youtube_channels_cleaned.csv")
